In [1]:
from common import *
import serial, csv, time
from datetime import datetime

In [2]:
def read_pipe_to_csv(arduino_port, stop_after=0, 
                     stop_below_temperature=30, uncertainty=0.0, 
                     debug_print=True):
    """
    Read data from Arduino over serial, write to CSV, and stop based on number of points
    or a minimum temperature threshold.
    
    Args:
        pipe_title (str): Title used in filename.
        arduino_port (str): Serial port, e.g., '/dev/ttyACM0'.
        stop_after (int): Number of datapoints to collect; 0 = unlimited.
        stop_below_temperature (float): Stop if temperature drops below this (in K).
        uncertainty (float): Measurement uncertainty to include in CSV.
        debug_print (bool): Print each line as it's collected.
    """
    # timestamped filename
    now = datetime.now()
    timestamp_str = now.strftime("%Y%m%d_%H%M%S")
    csv_filename = f"./data/test_{timestamp_str}.csv"
    
    print(f"Saving data to: {csv_filename}")
    
    # open serial port
    ser = serial.Serial(arduino_port, 9600, timeout=1)
    time.sleep(2)  # allow Arduino to reset

    # First datapoint is defined as t = 0 seconds
    seconds = 0
    
    try:
        with open(csv_filename, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Temperature 1 (K)', 'Temperature 2 (K)', 'Temperature 3 (K)', 'Uncertainty (K)', 'Time (s)'])
            
            while True:
                # stop based on number of points/seconds
                if stop_after > 0 and seconds >= stop_after:
                    print(f"Reached {stop_after} datapoints/seconds. Stopping.")
                    break
                
                # read line
                line_raw = ser.readline().decode('utf-8').strip()
                if not line_raw:
                    continue
                
                try:
                    temps = np.array([float(token) for token in line_raw.split(",")])
                    
                    # stop based on temperature
                    if np.all(temps < stop_below_temperature) and debug_print:
                        print(f"All temperatures are below {stop_below_temperature:.2f} C. Stopping.")
                        break

                    if debug_print:
                        if stop_after > 0:
                            print(f'{seconds}/{stop_after}', end=' ')
                        print(temps[0], 'C,', temps[1], 'C,', temps[2], 'C')

                    # save to CSV
                    temps = temps + 273.15 
                    writer.writerow([temps[0], temps[1], temps[2], uncertainty, seconds])
                    f.flush()
                    
                    seconds += 1
                    
                except ValueError:
                    continue
                    
    except KeyboardInterrupt:
        print("Stopped by user.")
        
    finally:
        ser.close()
        print("Serial port closed.")

From the MAX6675 K-Type Thermocouple Data Sheet, the uncertainty in the thermocouple at 5 volt VCC measurement between $0^\circ$ C and $+700^\circ $ C is $\pm 9/4 = \pm 2.25$ C/K [[1]](https://www.analog.com/media/en/technical-documentation/data-sheets/max6675.pdf).

In [3]:
read_pipe_to_csv('/dev/ttyACM0', stop_below_temperature=10, stop_after=100)

Saving data to: ./data/test_20260112_211911.csv
0/100 23.75 C, 23.25 C, 23.5 C
1/100 23.25 C, 22.75 C, 23.0 C
2/100 23.5 C, 23.25 C, 22.75 C
3/100 23.5 C, 23.5 C, 23.0 C
4/100 21.75 C, 22.0 C, 23.25 C
5/100 23.5 C, 23.5 C, 23.25 C
6/100 22.25 C, 22.5 C, 23.25 C
7/100 23.0 C, 23.0 C, 22.5 C
8/100 22.75 C, 23.25 C, 22.5 C
9/100 23.5 C, 23.25 C, 23.0 C
10/100 23.25 C, 23.5 C, 22.75 C
11/100 20.0 C, 23.25 C, 22.5 C
12/100 20.5 C, 23.0 C, 22.25 C
13/100 21.75 C, 23.25 C, 23.0 C
14/100 23.0 C, 23.25 C, 22.25 C
15/100 23.25 C, 22.5 C, 23.0 C
16/100 23.5 C, 22.75 C, 23.0 C
17/100 23.75 C, 23.0 C, 22.5 C
18/100 23.5 C, 23.5 C, 23.0 C
19/100 23.75 C, 23.25 C, 23.0 C
20/100 24.5 C, 23.0 C, 22.75 C
21/100 24.25 C, 23.5 C, 23.5 C
22/100 24.25 C, 23.5 C, 23.75 C
23/100 24.5 C, 23.5 C, 23.5 C
24/100 24.5 C, 23.5 C, 23.0 C
25/100 24.0 C, 23.5 C, 22.75 C
26/100 24.75 C, 23.25 C, 23.0 C
27/100 24.5 C, 23.25 C, 23.25 C
28/100 24.0 C, 23.25 C, 22.5 C
29/100 25.75 C, 23.5 C, 23.25 C
30/100 24.25 C, 22.75 C

SerialException: device reports readiness to read but returned no data (device disconnected or multiple access on port?)

In [ ]:
def load_max6675_csv_to_dataset(csv_filename):
    """
    Loads a CSV produced by the MAX6675 Arduino logger into a Dataset object
    using Python's built-in csv module.
    
    Args:
        csv_filename (str): Path to the CSV file.
        uncertainty (float, optional): Measurement uncertainty in K.
                                       If None, will try to read from CSV or default to 2 K.
    Returns:
        Dataset: x = time (s), y = temperature (K), dx = 0, dy = uncertainty
    """
    temperatures = [[], [], []]
    times = []
    dy_values = []
    
    with open(csv_filename, newline='') as csvfile:
        reader = csv.reader(csvfile)
        headers = next(reader)  # skip header
        # Identify columns
        try:
            temp_1_idx = headers.index('Temperature 1 (K)')
            temp_2_idx = headers.index('Temperature 2 (K)')
            temp_3_idx = headers.index('Temperature 3 (K)')
            uncert_idx = headers.index( 'Uncertainty (K)')
            time_idx   = headers.index('Time (s)')
        except ValueError:
            raise ValueError("CSV must have headers 'Temperature (K)',  'Uncertainty (K)' and 'Time (s)'")
        
        for row in reader:
            try:
                temp_1 = float(row[temp_1_idx])
                temp_2 = float(row[temp_2_idx])
                temp_3 = float(row[temp_3_idx])
                t  = float(row[time_idx])
                dy = float(row[uncert_idx])
            except ValueError:
                continue  # skip malformed rows
            
            temperatures[0].append(temp_1)
            temperatures[1].append(temp_2)
            temperatures[2].append(temp_3)
            times.append(t)
            dy_values.append(dy)

    datasets = [[],[],[]]
    for i in range(3):
        x = np.array(times)
        y = np.array(temperatures[i])
        dx = np.zeros_like(x)
        dy = np.array(dy_values)

        datasets[i] = Dataset(x=x, dx=dx, y=y, dy=dy)
    
    return datasets

In [ ]:
temperature_curve_gopts = GraphingOptions(
    x_label='Time',
    y_label='Temperature',
    x_units='s',
    y_units='K'
)

test_data = load_max6675_csv_to_dataset('./data/test_20260111_003028.csv')

In [ ]:
def plot_max6675_datasets(datasets, labels=None):
    """
    Plot multiple MAX6675 datasets on the same axes as lines (no uncertainties).
    
    Args:
        datasets (list of Dataset): List of Dataset objects
        labels (list of str, optional): Labels for each dataset
    """
    plt.figure()

    for i, ds in enumerate(datasets):
        label = labels[i] if labels is not None else f"TC {i+1}"
        plt.plot(ds.x, ds.y, label=label)

    plt.xlabel("Time (s)")
    plt.ylabel("Temperature (K)")
    plt.title("MAX6675 Temperature vs Time")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_max6675_datasets(test_data)

## 